<a href="https://colab.research.google.com/github/Mihan0207/Corporate_Bank_Loan_Automation/blob/main/Automated_Loan_Application_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Business Loan Application - Automated PDF Form Filler
**`Overview`**

This script automates the process of filling out business loan application PDFs using company data stored in a CSV file. Instead of manually typing company details into each form, the script looks up the company by its unique identifier, DBA (Doing Business As) name in this case and populates the relevant fields automatically — saving significant time when processing multiple applications.

**`What the Code Does`**

Reads Corporate Loan Database.csv, finds the company by DBA name, then fills two pages of the PDF template:


* Page 1 — Business Information: Legal name, DBA, primary contact, tax ID, telephone, address, business structure, and incorporation date
* Page 2 — Business Financials: Balance sheet items, liabilities, and income statement figures (revenue, COGS, net income)

The completed form is saved as a PDF named after the DBA name.


**`Why It Matters`**

Loan officers typically fill these forms manually — slow, repetitive, and error-prone. This script eliminates that:

1. Consistent — Data pulled directly from a verified source
2. Fast — A 40-50 manual task completed in under a minute
3. Scalable — Any company in the database can be processed instantly

This code cell is responsible for setting up a **Python virtual environment** and installing the `PyMuPDF` library. This library is crucial for working with PDF documents, including operations like opening, reading, and modifying them, which is required for filling out the loan application form.

In [2]:
!python3 -m venv venv
!source venv/bin/activate
!pip install PyMuPDF

Error: Command '['/content/venv/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.
/bin/bash: line 1: venv/bin/activate: No such file or directory
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 51.7 MB/s eta 0:00:00


In [3]:
!python -c "import fitz; doc = fitz.open(); doc.close(); print('✓ Working!')"

✓ Working!


This cell imports essential Python libraries:  
- `csv`: For reading and processing the company data from the CSV file.
- `os`: For interacting with the operating system, such as managing file paths and directories.
- `sys`: Provides access to system-specific parameters and functions, though less directly used for the core PDF generation logic in this script.
- `fitz` (PyMuPDF): The primary library for PDF manipulation, enabling the script to open, read, modify, and save PDF documents.  

These libraries collectively facilitate data handling, file management, and core PDF operations necessary for filling out and generating the loan application form.

In [4]:
import csv
import os
import sys
import fitz  # pymupdf

This code block defines the `SCRIPT_DIR` variable, which is crucial for consistently locating input files, specifically the CSV data file and the PDF template. It also ensures that the final generated PDF is saved in a reliable, script-relative location, making the script portable and robust.

In [5]:
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback for interactive environments where __file__ is not defined
    SCRIPT_DIR = os.getcwd()

This cell defines crucial constants:
- `CSV_PATH`: Specifies the file path for the CSV database containing corporate loan information.
- `PDF_TEMPLATE`: Specifies the file path for the blank PDF loan application form that will be filled.
- `FONT_NAME`, `FONT_SIZE`, `FONT_COLOR`: These set the stylistic properties for text inserted into the PDF, ensuring consistency and readability in the final document. They directly impact the appearance and accuracy of the filled-out PDF.
These constants ensure that the script can correctly locate its input files and format the output PDF consistently.

In [6]:
CSV_PATH = os.path.join(SCRIPT_DIR, "Corporate_Loan_Database.csv")
PDF_TEMPLATE = os.path.join(SCRIPT_DIR, "Bank Loan Form.pdf")

FONT_NAME = "Calibri"  # Calibri font
FONT_SIZE = 9
FONT_COLOR = (0, 0, 0)  # black



This cell defines two crucial utility functions for handling the company data from the CSV file:

*   **`load_csv()`**: This function is responsible for reading the entire `Corporate_Loan_Database.csv` file into memory. It parses the CSV content and returns it as a list of dictionaries, where each dictionary represents a company's record.

*   **`find_row_by_dba_name(rows, dba_name)`**: This function takes the loaded CSV data (`rows`) and a `DBA Name` as input. It efficiently searches through the company records to locate and return the specific row (company's data) that matches the provided `DBA Name`. This function is critical for retrieving the necessary company-specific information required to accurately populate the loan application form for the selected business.

These functions together form the data retrieval layer, ensuring that the correct company data is accessible for subsequent PDF filling operations.


In [7]:
def load_csv():

    with open(CSV_PATH, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        return list(reader)


def find_row_by_dba_name(rows, dba_name):

    dba_name_clean = dba_name.strip().lstrip("\t").lower()
    for row in rows:
        row_dba = row.get("DBA Name", "").strip().lstrip("\t").lower()
        if row_dba == dba_name_clean:
            return row
    return None


### Text Formatting and Placement Functions

This section defines two crucial utility functions: `clean_val()` and `put_text()`.

*   **`clean_val()`:** This function is designed to sanitize input data by handling potential `None` values and stripping leading/trailing whitespace. This ensures data consistency across the application and prevents errors when processing values from the CSV.

*   **`put_text()`:** This function is responsible for the precise placement and formatting of text on the PDF pages. It takes coordinates (x, y), the text content, font size, and an optional `max_width`. It attempts to use the specified `FONT_NAME` (Calibri) and falls back to Helvetica if Calibri is not available. Critically, it adjusts the font size dynamically if `max_width` is provided, ensuring that the text fits within the designated area. This function is vital for creating a well-formatted and readable final PDF document by controlling font, size, color, and position.

In [8]:
def clean_val(val):

    if val is None:
        return ""
    return val.strip().lstrip("\t")


def put_text(page, x, y, text, fontsize=FONT_SIZE, max_width=None):

    if not text:
        return

    try:
        font_name = FONT_NAME
        font = fitz.Font(font_name)
    except:
        font_name = "helv"
        font = fitz.Font(font_name)

    fs = fontsize
    if max_width:

        while fs > 3:
            tw = font.text_length(text, fontsize=fs)
            if tw <= max_width:
                break
            fs -= 0.25
    page.insert_text(
        fitz.Point(x, y),
        text,
        fontname=font_name,
        fontsize=fs,
        color=FONT_COLOR,
    )

### Filling Business Information

This function, `fill_business_info()`, is dedicated to populating the 'Business Information' section on the first page of the PDF loan application. It retrieves relevant data from the loaded company row (e.g., Legal Name, DBA Name, Tax ID, Address, Business Structure, Incorporation Date, Sector) and uses the `put_text()` function to place these details accurately onto the PDF template. This section provides crucial identification and structural details of the applicant company, which are essential for the loan application.

In [15]:
def fill_business_info(page, row):


    # Business Legal Name
    put_text(page, 35, 482, clean_val(row.get("Legal Name (Yahoo)")), max_width=540)

    # Doing Business as (DBA) Name
    put_text(page, 35, 504, clean_val(row.get("DBA Name")), max_width=180)

    # Primary Contact
    put_text(page, 280, 504, clean_val(row.get("Primary Contact")), max_width=230)

    # Tax I.D. #
    put_text(page, 35, 527, clean_val(row.get("Tax ID (Company No)")), max_width=205)

    # Telephone
    put_text(page, 280, 527, clean_val(row.get("Telephone")), max_width=140)

    # Street Address
    put_text(page, 35, 548, clean_val(row.get("Registered Address")), max_width=270)

    # City
    put_text(page, 280, 548, clean_val(row.get("City")), max_width=112)

    # State
    state = clean_val(row.get("State/Province (Jurisdiction)"))
    put_text(page, 420, 548, state, max_width=55, fontsize=9)

    # Zip Code
    put_text(page, 510, 548, clean_val(row.get("Zip/Post Code")), max_width=55, fontsize=9)

    # Business Structure
    structure = clean_val(row.get("Business Structure", "")).lower()

    # Check if structure is plc - tick C Corporation
    if "plc" in structure:
        put_text(page, 45, 588, "X", fontsize=9)
    else:
        put_text(page, 456, 598, "X", fontsize=7)
        put_text(page, 490, 598, clean_val(row.get("Business Structure", "")), max_width=70, fontsize=6)

    # Date Business Established: Month ___ Year ___
    inc_date = clean_val(row.get("Incorporation Date", ""))
    if inc_date:
        parts = inc_date.split("-")
        if len(parts) >= 2:
            month = parts[1]
            year = parts[0]
            put_text(page, 135, 614, month)
            put_text(page, 175, 614, year)

    # State of Incorporation
    put_text(page, 302, 610, state, max_width=60, fontsize=6)

    # Explain Nature of Business
    put_text(page, 127, 628, clean_val(row.get("Sector", "")), max_width=430)

###Fill_business_financials

This function is responsible for populating the financial details on the second(2nd) page of the PDF loan application. It retrieves various financial data from the company's data row, including balance sheet information, income statement dates, cash, accounts receivable, inventory, machinery/equipment, real estate, accounts payable, notes payable, mortgages, gross sales/revenue, cost of goods sold, and business net income/loss. The function then utilizes the `put_text()` helper function to accurately place these values onto the PDF template. This section is a key component of the completed loan application as it provides a comprehensive overview of the applicant's financial health.

In [16]:

def fill_business_financials(page, row):


    bs_date = clean_val(row.get("Balance Sheet Date", ""))
    if bs_date:
        parts = bs_date.split("-")
        if len(parts) == 3:
            put_text(page, 90, 474, f"{parts[1]} / {parts[2]} / {parts[0]}")


    is_date = clean_val(row.get("Income Statement Date", ""))
    if is_date:
        parts = is_date.split("-")
        if len(parts) == 3:
            put_text(page, 490, 474, f"{parts[1]} / {parts[2]} / {parts[0]}")


    ax = 120

    row_y = [492, 509, 524, 540, 556, 571, 586]
    # Cash
    put_text(page, ax, row_y[0], clean_val(row.get("Cash")), max_width=70, fontsize=8)
    # Accounts Receivable
    put_text(page, ax, row_y[1], clean_val(row.get("Accounts Receivable")), max_width=70, fontsize=8)
    # Inventory
    put_text(page, ax, row_y[2], clean_val(row.get("Inventory")), max_width=70, fontsize=8)
    # Machinery/Equipment
    put_text(page, ax, row_y[3], clean_val(row.get("Machinery/Equipment")), max_width=70, fontsize=8)
    # Automobiles – not in CSV
    # Real Estate
    put_text(page, ax, row_y[5], clean_val(row.get("Real Estate")), max_width=70, fontsize=8)



    mx = 270
    # Accounts Payable
    put_text(page, mx, row_y[0], clean_val(row.get("Accounts Payable")), max_width=75, fontsize=8)
    # Notes Payable
    put_text(page, mx, row_y[1], clean_val(row.get("Notes Payable (Short Term Debt)")), max_width=75, fontsize=8)

    # Automotive Loans – not in CSV
    # Mortgages
    put_text(page, mx, row_y[4], clean_val(row.get("Mortgages (Long Term Debt)")), max_width=75, fontsize=8)

    rx = 485
    # GROSS SALES/REVENUE (+)
    put_text(page, rx, row_y[0], clean_val(row.get("Revenue")), max_width=100, fontsize=8)
    # Cost of Goods Sold (-)
    put_text(page, rx, row_y[1], clean_val(row.get("Cost of Goods Sold")), max_width=100, fontsize=8)

    # Interest Expense – not in CSV
    # Depreciation – not in CSV
    # BUSINESS NET INCOME / (NET LOSS) (=)
    put_text(page, rx, row_y[6], clean_val(row.get("Net Income")), max_width=100, fontsize=8)




### Main Execution Logic

This `main` function serves as the central orchestrator for the entire PDF filling process. It coordinates all the previously defined utility and filling functions to generate a personalized loan application document. The workflow is as follows:

1.  **Data Loading and Validation**: It first checks for the existence of the `Corporate_Loan_Database.csv` and `Bank Loan Form.pdf` files. If either is missing, it prints an error and exits. Otherwise, it loads all company data from the CSV file using the `load_csv()` function.
2.  **User Input**: It prompts the user to enter a 'DBA Name' to identify the specific company for which the form needs to be filled.
3.  **Company Lookup**: Using the provided DBA Name, it searches for the corresponding company's data row in the loaded CSV data via the `find_row_by_dba_name()` function.
4.  **PDF Template Opening**: If a company is found, it opens the `Bank Loan Form.pdf` template using `fitz.open()`.
5.  **Form Filling**: It then calls `fill_business_info()` to populate the first page (index 0) with business details and `fill_business_financials()` to fill the second page (index 1) with financial information, passing the relevant company data row to each.
6.  **Saving the Output**: Finally, it constructs a safe filename based on the company's DBA Name, saves the completed PDF to a new file in the `SCRIPT_DIR`, and closes the document.

In [17]:

def main():
    # Load data
    if not os.path.exists(CSV_PATH):
        print(f"Error: CSV file not found at {CSV_PATH}")
        sys.exit(1)
    if not os.path.exists(PDF_TEMPLATE):
        print(f"Error: PDF template not found at {PDF_TEMPLATE}")
        sys.exit(1)

    rows = load_csv()
    print(f"Loaded {len(rows)} rows from CSV.")

    # Getting DBA Name from user
    dba_name = input("Enter DBA Name: ").strip()
    if not dba_name:
        print("Error: No DBA Name provided.")
        sys.exit(1)

    # Looking up the row
    row = find_row_by_dba_name(rows, dba_name)
    if row is None:
        print(f"Error: No matching row found for DBA Name '{dba_name}'.")
        sys.exit(1)

    company_name = clean_val(row.get("Legal Name (Yahoo)", "Unknown"))
    tax_id = clean_val(row.get("Tax ID (Company No)", "Unknown"))
    dba = clean_val(row.get("DBA Name", tax_id))  # Using DBA primarily, fallback to Tax ID
    print(f"Found: {company_name}")
    print(f"Tax ID: {tax_id}")

    doc = fitz.open(PDF_TEMPLATE)

    # Fill page 0 – Business Information
    fill_business_info(doc[0], row)

    # Fill page 1 – Business Financials
    fill_business_financials(doc[1], row)

    # Save using DBA Name
    safe_dba = dba.replace("\t", "").replace(" ", "_").replace("/", "-").replace("\\", "-")
    output_path = os.path.join(SCRIPT_DIR, f"{safe_dba}.pdf")
    doc.save(output_path)
    doc.close()

    print(f"Saved filled PDF to: {output_path}")


if __name__ == "__main__":
    main()

Loaded 100 rows from CSV.
Enter DBA Name: Tesco
Found: Tesco PLC
Tax ID: 00445790
Saved filled PDF to: /content/Tesco.pdf


#Key Finding and Summary

**What We Built**

A end-to-end pipeline that automatically sources, structures, and delivers business intelligence for corporate loan applications. The model pulls live data from two authoritative sources — Yahoo Finance (financials) and Companies House API (legal registry) — merges them into a single verified dataset, and auto-populates loan application PDFs on demand.

**Business Value**
The model compresses what would be hours of manual research and form completion per application into a sub-2-second automated workflow. For a lending team processing dozens of corporate applications monthly, this translates directly into:

1. Reduced operational cost per application
2. Lower risk of transcription errors in financial figures
3. A consistent, auditable data trail from source to submitted form
4. Faster credit decision cycles by removing the data-gathering bottleneck


**Summary**

The pipeline demonstrates that publicly available financial and regulatory data — when properly sourced, cleaned, and structured — is sufficient to automate the majority of a corporate loan application. The remaining gaps (email, real-time financials, private companies) point toward natural extensions: integrating a paid enrichment API (e.g. Clearbit, Bloomberg), adding a provincial Canadian registry layer via the ISED key, and building a multi-template PDF engine that adapts to different lenders' forms without coordinate recalibration.